# A Claude chatbot

A multi-turn chatbot built directly on the Messages API with the official `anthropic` SDK.

It runs on **Claude Haiku 4.5**, the cheapest model — $1 per million input tokens, $5 per million output. A long session here costs a fraction of a cent.

**Setup (once):**

1. `pip install -r requirements.txt`
2. `cp .env.example .env`
3. Paste your key into `.env` (get one at [platform.claude.com/settings/keys](https://platform.claude.com/settings/keys))

`.env` is listed in `.gitignore`, so the key stays on your machine. Never paste the key into a notebook cell — cell contents get committed.

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()  # reads .env from this folder into the environment

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found. Copy .env.example to .env and paste your key in."
    )

# The client reads ANTHROPIC_API_KEY from the environment - don't pass it explicitly.
client = anthropic.Anthropic()

MODEL = "claude-haiku-4-5"  # cheapest model: $1 / $5 per million tokens
print("Client ready ->", MODEL)

## 1. One message, one response

The smallest possible call. `content` is a *list of blocks*, not a string, so check `block.type` before reading `.text`.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "In one sentence, what is the Claude API?"}],
)

for block in response.content:
    if block.type == "text":
        print(block.text)

print("\nstop_reason:", response.stop_reason)
print("tokens:", response.usage.input_tokens, "in /", response.usage.output_tokens, "out")

## 2. The chatbot

The Messages API is **stateless** — it has no memory of previous calls. To hold a conversation you keep the history yourself and resend it every turn. That's the whole trick; everything below is bookkeeping around it.

A few choices worth knowing about:

- **Streaming** (`client.messages.stream`) prints tokens as they arrive instead of waiting for the full response. Use it for anything user-facing.
- **`effort`** (`low` → `max`) controls how hard the model thinks before answering. It's a Claude 4.6+ feature — **Haiku 4.5 doesn't have it, and sending it returns a 400** — so `Chatbot` only passes it when you set one explicitly.
- **Full content blocks go back into the history**, not just the text. Haiku 4.5 doesn't emit thinking blocks, but doing it this way keeps the code correct if you later switch to a model that does.

In [ ]:
class Chatbot:
    """A multi-turn conversation with Claude.

    chat() streams the reply straight to the cell output and stores it on
    .last_reply. History lives in .messages and is resent on every turn.
    """

    def __init__(
        self,
        system: str | None = None,
        model: str = MODEL,
        effort: str | None = None,  # Claude 4.6+ only: low|medium|high|xhigh|max
        max_tokens: int = 4096,
    ):
        self.system = system
        self.model = model
        self.effort = effort
        self.max_tokens = max_tokens
        self.messages: list[dict] = []
        self.last_reply: str = ""
        self.input_tokens = 0
        self.output_tokens = 0

    def chat(self, user_message: str) -> None:
        self.messages.append({"role": "user", "content": user_message})

        params = {
            "model": self.model,
            "max_tokens": self.max_tokens,
            "messages": self.messages,
        }
        if self.system:
            params["system"] = self.system
        if self.effort:  # omitted by default - Haiku 4.5 has no effort parameter
            params["output_config"] = {"effort": self.effort}

        with client.messages.stream(**params) as stream:
            for chunk in stream.text_stream:
                print(chunk, end="", flush=True)
            final = stream.get_final_message()
        print()

        # Append the response's content blocks verbatim, not just the text.
        self.messages.append({"role": "assistant", "content": final.content})

        self.last_reply = "".join(b.text for b in final.content if b.type == "text")
        self.input_tokens += final.usage.input_tokens
        self.output_tokens += final.usage.output_tokens

        if final.stop_reason == "max_tokens":
            print("\n[truncated - raise max_tokens]")

    def reset(self) -> None:
        """Forget the conversation. Keeps the system prompt and settings."""
        self.messages.clear()
        self.last_reply = ""

    def usage(self) -> dict:
        return {
            "turns": len(self.messages) // 2,
            "input_tokens": self.input_tokens,
            "output_tokens": self.output_tokens,
        }


bot = Chatbot()
print("bot ready - call bot.chat('...')")

## 3. Talk to it

Run these two cells in order — the second one only works because the first turn is still in the history.

In [ ]:
bot.chat("Hi! I'm Jubaer, and I'm learning the Claude API today.")

In [ ]:
bot.chat("What's my name, and what am I working on?")

In [ ]:
# Your turn - edit and rerun this cell as many times as you like.
bot.chat("Explain prompt caching to me like I've been coding for ten years.")

## 4. Steering it

The **system prompt** sets persona, rules, and output format. It isn't part of the message history — it's a separate field that applies to the whole conversation.

`reset()` clears the history without throwing away the settings.

In [ ]:
reviewer = Chatbot(
    system=(
        "You are a blunt senior Python reviewer. Point out real bugs and "
        "design problems only - skip style nits. Keep it under 120 words."
    ),
    # effort="medium",   # only valid if you switch MODEL to Sonnet 5 / Opus 5
)

reviewer.chat(
    "Review this:\n\n"
    "def get_user(users, uid):\n"
    "    for u in users:\n"
    "        if u['id'] == uid:\n"
    "            return u\n"
)

In [ ]:
bot.reset()
bot.chat("Do you remember my name?")   # history is gone, so: no

## 5. What it cost

Input tokens grow every turn, because the whole history is resent each time. That's the main cost driver in a long conversation — and the reason [prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) exists.

Rates below are Claude Haiku 4.5 list pricing. If you change `MODEL`, change these too — check the [pricing page](https://platform.claude.com/docs/en/pricing) for current numbers.

In [ ]:
IN_PER_MTOK, OUT_PER_MTOK = 1.00, 5.00  # USD, claude-haiku-4-5

for name, b in [("bot", bot), ("reviewer", reviewer)]:
    u = b.usage()
    cost = (u["input_tokens"] * IN_PER_MTOK + u["output_tokens"] * OUT_PER_MTOK) / 1e6
    print(f"{name:9} {u['turns']:>2} turns  "
          f"{u['input_tokens']:>7} in  {u['output_tokens']:>6} out  ~${cost:.5f}")

## Where to go next

- **More capability:** swap `MODEL` to `claude-sonnet-5` ($3/$15) or `claude-opus-5` ($5/$25). Both support `effort`, so you can then pass `effort="high"` to `Chatbot` for harder reasoning — and update the price constants above.
- **Give it hands:** [tool use](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview) lets Claude call your functions; the SDK's tool runner drives the loop for you.
- **Long conversations:** [prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) cuts the cost of resending history.
- **Typed responses:** [structured outputs](https://platform.claude.com/docs/en/build-with-claude/structured-outputs) constrain replies to a JSON schema.
- **Bulk work:** the [Batch API](https://platform.claude.com/docs/en/build-with-claude/batch-processing) halves the price again for anything not latency-sensitive.